In [6]:
# ACTIVITY1 WEB SERVER ACCESS LOG ANALYSIS
import csv
import json
import shutil
from collections import Counter
import os

output_folder = "activity1-data"
os.makedirs(output_folder, exist_ok=True)

ID_SUM = 25
SEED_DIGIT = 7

LIMIT = 1000 + ID_SUM
TARGET = str((SEED_DIGIT % 4) + 2)

INPUT = "weblog.csv"
FILTERED = os.path.join(output_folder, "filtered_log.csv")
SUMMARY = os.path.join(output_folder, "summary.json")
BACKUP = os.path.join(output_folder, "summary_backup.json")

processed = 0
selected = []
status_count = Counter()
url_count = Counter()

with open(INPUT, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    for row in reader:
        if processed >= LIMIT:
            break

        processed += 1

        try:
            ip = row["IP"]
            time = row["Time"]
            url = row["URL"]
            status = row["Staus"].strip()

            if status.startswith(TARGET) and status.isdigit():
                selected.append(row)
                status_count[status] += 1
                url_count[url] += 1

        except (KeyError, AttributeError, TypeError):
            continue

# Save filtered records
with open(FILTERED, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=reader.fieldnames)
    writer.writeheader()
    writer.writerows(selected)

# Create summary
summary = {
    "Records Processed": processed,
    "Target Status Class": TARGET + "xx",
    "Records in Target Class": len(selected),
    "Count per Status Code": dict(status_count),
    "Five Most Requested URLs": url_count.most_common(5),
    "Filtered CSV File": FILTERED
}

with open(SUMMARY, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

# Create and verify backup
shutil.copy(SUMMARY, BACKUP)

print("Records Processed:", processed)
print("Target Status Class:", TARGET + "xx")
print("Records in Target Class:", len(selected))
print("Count per Status Code:", dict(status_count))
print("Top 5 URLs:", url_count.most_common(5))
print("Filtered CSV:", FILTERED)
print("JSON Summary:", SUMMARY)
print("Backup Verified:", 
      "Yes" if __import__("os").path.exists(BACKUP) else "No")

Records Processed: 1025
Target Status Class: 5xx
Records in Target Class: 0
Count per Status Code: {}
Top 5 URLs: []
Filtered CSV: activity1-data\filtered_log.csv
JSON Summary: activity1-data\summary.json
Backup Verified: Yes


In [4]:
# ACTIVITY2 MOBILE DEVISE USAGE PROFILING

import csv
import json
import os
import shutil

SEED_DIGIT = 7

output_folder = "activity2-data"
os.makedirs(output_folder, exist_ok=True)

target_os = "Android" if SEED_DIGIT % 2 == 0 else "iOS"
target_class = (SEED_DIGIT % 5) + 1

input_file = "user_behavior_dataset.csv"
output_file = os.path.join(output_folder, "filtered_mobile_usage.csv")
summary_file = os.path.join(output_folder, "mobile_usage_summary.json")
backup_file = os.path.join(output_folder, "filtered_mobile_usage_backup.csv")

total_records = 0
os_records = 0
selected_records = []

battery_total = 0
data_total = 0

with open(input_file, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    with open(output_file, "w", newline="", encoding="utf-8") as out:
        fields = [
            "User ID",
            "Device Model",
            "Operating System",
            "Battery Drain (mAh/day)",
            "Data Usage (MB/day)"
        ]

        writer = csv.DictWriter(out, fieldnames=fields)
        writer.writeheader()

        for row in reader:
            total_records += 1

            if row["Operating System"].strip() != target_os:
                continue

            os_records += 1

            try:
                if int(row["User Behavior Class"]) != target_class:
                    continue

                battery = float(row["Battery Drain (mAh/day)"])
                data = float(row["Data Usage (MB/day)"])

                selected_records.append(row)
                battery_total += battery
                data_total += data

                writer.writerow({field: row[field] for field in fields})

            except (ValueError, KeyError):
                continue

count = len(selected_records)

avg_battery = battery_total / count if count else 0
avg_data = data_total / count if count else 0

with open(output_file, "r", newline="", encoding="utf-8") as file:
    exported_count = sum(1 for row in csv.DictReader(file))

count_verified = exported_count == count

shutil.copy2(output_file, backup_file)
backup_size = os.path.getsize(backup_file)

summary = {
    "operating_system": target_os,
    "target_behavior_class": target_class,
    "total_records": total_records,
    "matching_os_records": os_records,
    "selected_records": count,
    "average_battery_drain": round(avg_battery, 2),
    "average_data_usage": round(avg_data, 2),
    "exported_records": exported_count,
    "count_verification": count_verified,
    "exported_csv": output_file,
    "backup_file": backup_file,
    "backup_size_bytes": backup_size
}

with open(summary_file, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

print("\nMOBILE DEVICE USAGE REPORT")
print("-------------------------")
print("Operating System:", target_os)
print("Target Behavior Class:", target_class)
print("Total Records:", total_records)
print("Matching OS Records:", os_records)
print("Selected Records:", count)
print("Average Battery Drain:", round(avg_battery, 2))
print("Average Data Usage:", round(avg_data, 2))
print("Count Verification:", count_verified)
print("Exported CSV:", output_file)
print("JSON Summary:", summary_file)
print("Backup File:", backup_file)
print("Backup Size:", backup_size, "bytes")



MOBILE DEVICE USAGE REPORT
-------------------------
Operating System: iOS
Target Behavior Class: 3
Total Records: 700
Matching OS Records: 146
Selected Records: 32
Average Battery Drain: 1492.03
Average Data Usage: 828.09
Count Verification: True
Exported CSV: activity2-data\filtered_mobile_usage.csv
JSON Summary: activity2-data\mobile_usage_summary.json
Backup File: activity2-data\filtered_mobile_usage_backup.csv
Backup Size: 973 bytes


In [13]:
# ACTIVITY3 CYBERSECURITY INCIDENT REPORT

import csv
import json
import os
import shutil
import filecmp

SEED_DIGIT = 7
TARGET_YEAR = 2015 + SEED_DIGIT

input_file = "Global_Cybersecurity_Threats_2015-2024.csv"

output_folder = "activity3-data"
os.makedirs(output_folder, exist_ok=True)

filtered_file = os.path.join(output_folder, "filtered_incidents.csv")
json_file = os.path.join(output_folder, "top_5_incidents.json")
backup_file = os.path.join(output_folder, "top_5_backup.json")
report_file = os.path.join(output_folder, "incident_report.txt")
summary_file = os.path.join(output_folder, "incident_summary.json")

total_records = 0
selected_records = []
attack_counts = {}
valid_losses = []

with open(input_file, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    if not reader.fieldnames:
        raise ValueError("Dataset is empty or missing headers.")

    with open(filtered_file, "w", newline="", encoding="utf-8") as out:
        writer = csv.DictWriter(out, fieldnames=reader.fieldnames)
        writer.writeheader()

        for row in reader:
            total_records += 1

            try:
                if int(row["Year"]) != TARGET_YEAR:
                    continue
            except (ValueError, TypeError, KeyError):
                continue

            selected_records.append(row)
            writer.writerow(row)

            # Count incidents per attack type
            attack = row.get("Attack Type", "").strip() or "Unknown"
            attack_counts[attack] = attack_counts.get(attack, 0) + 1

            # Process valid financial losses
            try:
                loss = float(row["Financial Loss (in Million $)"])

                if loss < 0:
                    continue

                valid_losses.append((loss, row))

            except (ValueError, TypeError, KeyError):
                continue

total_loss = sum(loss for loss, row in valid_losses)
valid_count = len(valid_losses)

average_loss = total_loss / valid_count if valid_count else 0

top_five = sorted(
    valid_losses,
    key=lambda item: -item[0]
)[:5]

top_five_records = []

for loss, row in top_five:
    incident = row.copy()
    incident["Financial Loss (in Million $)"] = loss
    top_five_records.append(incident)

with open(json_file, "w", encoding="utf-8") as file:
    json.dump(top_five_records, file, indent=4)

shutil.copy2(json_file, backup_file)
backup_verified = filecmp.cmp(json_file, backup_file, shallow=False)

with open(filtered_file, "r", newline="", encoding="utf-8") as file:
    exported_count = sum(1 for row in csv.DictReader(file))

count_verified = exported_count == len(selected_records)

summary_file = os.path.join(
    output_folder, "incident_summary.json"
)

summary = {
    "target_year": TARGET_YEAR,
    "total_records": total_records,
    "incidents_in_target_year": len(selected_records),
    "count_per_attack_type": attack_counts,
    "total_financial_loss": round(total_loss, 2),
    "average_financial_loss": round(average_loss, 2),
    "five_costliest_incidents": top_five_records,
    "filtered_csv": filtered_file,
    "top_five_json": json_file,
    "backup_verified": backup_verified,
    "csv_count_verified": count_verified
}

with open(summary_file, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

report = [
    "CYBERSECURITY INCIDENT REPORT",
    "-----------------------------",
    f"Target Year: {TARGET_YEAR}",
    f"Total Dataset Records: {total_records}",
    f"Incidents in Target Year: {len(selected_records)}",
    "",
    "INCIDENT COUNT PER ATTACK TYPE"
]

for attack, count in sorted(attack_counts.items()):
    report.append(f"{attack}: {count}")

report.extend([
    "",
    f"Total Financial Loss: {total_loss:.2f} million USD",
    f"Average Financial Loss: "
    f"{total_loss / valid_count:.2f} million USD"
    if valid_count else "Average Financial Loss: N/A",
    "",
    "FIVE COSTLIEST INCIDENTS"
])

for i, incident in enumerate(top_five_records, 1):
    report.append(
        f"{i}. {incident['Attack Type']} - "
        f"{incident['Financial Loss (in Million $)']} million USD"
    )

report.extend([
    "",
    f"Filtered CSV: {filtered_file}",
    f"JSON File: {json_file}",
    f"Backup Verified: {backup_verified}",
    f"CSV Count Verified: {count_verified}",
    f"Report File: {report_file}"
])

with open(report_file, "w", encoding="utf-8") as file:
    file.write("\n".join(report))

print("\n".join(report))
print("Backup File:", backup_file)
print("Backup Size:", os.path.getsize(backup_file), "bytes")


CYBERSECURITY INCIDENT REPORT
-----------------------------
Target Year: 2022
Total Dataset Records: 3000
Incidents in Target Year: 318

INCIDENT COUNT PER ATTACK TYPE
DDoS: 54
Malware: 64
Man-in-the-Middle: 41
Phishing: 57
Ransomware: 56
SQL Injection: 46

Total Financial Loss: 15870.86 million USD
Average Financial Loss: 49.91 million USD

FIVE COSTLIEST INCIDENTS
1. Phishing - 98.47 million USD
2. SQL Injection - 98.46 million USD
3. Phishing - 98.06 million USD
4. Phishing - 97.56 million USD
5. Malware - 97.32 million USD

Filtered CSV: activity3-data\filtered_incidents.csv
JSON File: activity3-data\top_5_incidents.json
Backup Verified: True
CSV Count Verified: True
Report File: activity3-data\incident_report.txt
Backup File: activity3-data\top_5_backup.json
Backup Size: 2221 bytes
